# Tomato instance segmentation — YOLO11n-seg on Laboro dataset

Training a nano segmentation model for eventual deployment on OAK-D Lite (Myriad X). Three ripeness classes: `fully_ripened`, `green`, `half_ripened`.

In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics roboflow blobconverter onnxsim -q

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="_")
project = rf.workspace("jalals-lab").project("laboro-tomato-kwpth")
dataset = project.version(2).download("yolov8", location="/kaggle/working/datasets/Laboro_Tomato")

print(dataset.location)

In [ ]:
import yaml

dataset_path = "/kaggle/working/datasets/Laboro_Tomato"

data_config = {
    "path": dataset_path,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": {
        0: "fully_ripened",
        1: "green",
        2: "half_ripened"
    }
}

config_path = "/kaggle/working/data_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(data_config, f)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")

results = model.train(
    data=config_path,
    epochs=100,
    imgsz=416,
    batch=32,
    device=0,
    project="/kaggle/working/tomato_segmentation",
    name="training_run",
    save=True,
    plots=True,
    amp=True,
    patience=20,
    save_period=10,
)

In [ ]:
best_model = YOLO("/kaggle/working/tomato_segmentation/training_run/weights/best.pt")
metrics = best_model.val(data=config_path)

print(f"Box  mAP@0.5      {metrics.box.map50:.3f}")
print(f"Box  mAP@0.5:0.95 {metrics.box.map:.3f}")
print(f"Mask mAP@0.5      {metrics.seg.map50:.3f}")
print(f"Mask mAP@0.5:0.95 {metrics.seg.map:.3f}")

In [ ]:
import glob
import matplotlib.pyplot as plt

val_images = glob.glob("/kaggle/working/datasets/Laboro_Tomato/valid/images/*.jpg")[:4]
preds = best_model(val_images, conf=0.25, imgsz=416)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (r, ax) in enumerate(zip(preds, axes.flat)):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.axis("off")
    ax.set_title(f"{len(r.boxes)} detections")
plt.tight_layout()
plt.savefig("/kaggle/working/sample_predictions.png", dpi=150)
plt.show()

In [ ]:
onnx_path = best_model.export(
    format="onnx",
    imgsz=416,
    opset=12,
    simplify=True,
    half=False,
)

print("ONNX:", onnx_path)

In [ ]:
import blobconverter

blob_path = blobconverter.from_onnx(
    model=onnx_path,
    data_type="FP16",
    shaves=6,
    use_cache=False,
    output_dir="/kaggle/working",
)

print("Blob:", blob_path)

In [ ]:
import shutil, os

weights_dir = "/kaggle/working/tomato_segmentation/training_run/weights"
out_dir = "/kaggle/working"

shutil.copy(f"{weights_dir}/best.pt", f"{out_dir}/best.pt")
shutil.copy(f"{weights_dir}/last.pt", f"{out_dir}/last.pt")
shutil.copy(onnx_path, f"{out_dir}/best.onnx")

for f in ["best.pt", "last.pt", "best.onnx", os.path.basename(blob_path)]:
    p = f"{out_dir}/{f}"
    print(f"{f:<20} {os.path.getsize(p)/1e6:.1f} MB")